# Tier 3 — 2/3: ajuste final dos Transformers (10 sementes)

Hiperparâmetros congelados da sessão 1, 200 épocas / paciência 16 com o melhor checkpoint restaurado.
Retomável semente a semente.

**Como rodar:** Settings → Accelerator → **GPU T4 x2**, Internet **ligada** → **Save Version → Save & Run All**.
O runner para sozinho ao atingir 10 h (nenhum trial/semente novo começa depois disso) e tudo o que terminou
fica em `/kaggle/working/tier3_out/`, que vira a saída da versão salva.

**Entrada obrigatória:** a saída da sessão 1 (**Add Input → Your Work → Notebook Output →
tier3_1_selecao**). Se esta sessão não terminar, nova versão com a própria saída como entrada (além da
seleção) e Save & Run All.

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2'

In [ ]:
# ── 2. Repositório ──
import os, subprocess, sys
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
sys.path.insert(0, '.')
assert 'TIER3_DEADLINE' in open('scripts/run_tier3.py').read(), 'clone sem prazo no runner — refaça o pull'

In [ ]:
# ── 3. Dependências e dados (em cache no clone; nada é baixado) ──
subprocess.run('pip install -q einops optuna', shell=True)
from pathlib import Path
from src.data.loaders import DatasetLoader
for d in ['HIGGS500K', 'COVER', 'KDD99']:
    assert Path(f'data/raw/{d}.parquet').exists(), f'sem {d}.parquet no clone — refaça o pull'
    X, y, _ = DatasetLoader.load(d)
    print(f'{d:10s} {X.shape}  positivos {y.mean():.3f}')

In [ ]:
# ── 4. Configuração, retomada automática e execução nas duas placas ──
import json, shutil, time
from pathlib import Path

# Prazo: nenhum trial/semente novo começa depois de 10 h (margem para o trial em curso
# e para o Kaggle salvar a saída antes das 12 h). O que terminou fica em tier3_out/.
DEADLINE_H = 10.0
os.environ['TIER3_DEADLINE'] = str(time.time() + DEADLINE_H * 3600)

OUTDIR = Path('/kaggle/working/tier3_out'); OUTDIR.mkdir(exist_ok=True)
# Retomada: copia qualquer tier3_* das entradas (saída de uma sessão anterior
# adicionada via "Add Input" → Notebook Output). Nada a configurar.
for f in sorted(Path('/kaggle/input').rglob('tier3_*')):
    if f.is_file() and not (OUTDIR / f.name).exists():
        shutil.copy(f, OUTDIR / f.name); print('retomado:', f)

DATASETS = 'HIGGS500K COVER KDD99'
SEEDS    = list(range(10))
# Dos quatro FT só ficam o Softmax (referência densa) e o Top-k (o esparso mais barato):
# os quatro empatam nos Tiers 1 e 2. O SAINT (atenção B×B) é o mais caro e fica sozinho.
GPU_MODELS = {0: ['FTTransformer_softmax', 'FTTransformer_topk', 'FTTransformerCURRandom'],
              1: ['SAINTColnorm']}

def files(gpu, tag=''):
    return (f'--selection-file {OUTDIR}/tier3_selection{tag}_gpu{gpu}.json '
            f'--final-file {OUTDIR}/tier3_final{tag}_gpu{gpu}.json '
            f'--storage sqlite:///{OUTDIR}/tier3_optuna{tag}_gpu{gpu}.db')

def run_2gpu(make_cmd, name):
    """Um processo por placa, log próprio em tier3_out/; espera os dois."""
    procs = []
    for gpu in (0, 1):
        log = open(OUTDIR / f'tier3_{name}_gpu{gpu}.log', 'a')
        cmd = f'CUDA_VISIBLE_DEVICES={gpu} python -u ' + make_cmd(gpu)
        print(f'[gpu{gpu}]', cmd, flush=True)
        procs.append((gpu, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT), log))
    t0 = time.time()
    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(300)
        print(f'{(time.time()-t0)/3600:.2f} h —', ' | '.join(
            f'gpu{g}: ' + ((open(l.name).read().strip().splitlines() or [''])[-1][-110:]) for g, _, l in procs), flush=True)
    for g, p, l in procs:
        l.close(); print(f'gpu{g} terminou com código {p.returncode}')

In [ ]:
# ── 5. Confere a seleção ──
for g in (0, 1):
    f = OUTDIR / f'tier3_selection_gpu{g}.json'
    assert f.exists(), f'sem {f.name} — adicione a saída da sessão 1 como entrada'
    sel = json.load(open(f))
    inc = [(v, d) for v in GPU_MODELS[g] for d in DATASETS.split() if not sel.get(v, {}).get(d, {}).get('complete')]
    assert not inc, f'seleção incompleta: {inc} — termine a sessão 1'
print('seleção completa')

In [ ]:
# ── 6. Ajuste final ──
run_2gpu(lambda g: 'scripts/run_tier3.py final --variants ' + ' '.join(GPU_MODELS[g]) +
         ' --datasets ' + DATASETS + ' --seeds ' + ' '.join(map(str, SEEDS)) + ' ' + files(g), 'final')

In [ ]:
# ── 7. Situação ──
import collections, statistics as stt
recs = []
for g in (0, 1):
    f = OUTDIR / f'tier3_final_gpu{g}.json'
    if f.exists(): recs += json.load(open(f))
by = collections.defaultdict(list)
for r in recs:
    if r.get('status') == 'ok': by[(r['variant'], r['dataset'])].append(r['test_f1_macro'])
falta = 0
for g in (0, 1):
    for v in GPU_MODELS[g]:
        for d in DATASETS.split():
            xs = by.get((v, d), []); falta += len(SEEDS) - len(xs)
            print(f'{v:26s} {d:10s} {len(xs):2d}/{len(SEEDS)}' +
                  (f'  F1 {stt.mean(xs):.4f} ± {stt.pstdev(xs):.4f}' if xs else ''))
erros = [r for r in recs if r.get('status') != 'ok']
print(f'\n{len(erros)} erros', [(r['variant'], r['dataset'], r['seed'], r['error'][:80]) for r in erros][:5])
print('FINAL COMPLETO — me passe tier3_out/.' if not falta else f'{falta} execuções pendentes — rode de novo com esta saída como entrada.')